# TP 7 — MLOps : pipeline, persistance, *skew* et dérive

> Support théorique : [cours_mlops_production.md](cours_mlops_production.md).

**Objectif.** Rendre concrets les points du module 7 : construire une **Pipeline sans fuite**, **sérialiser**
puis recharger un modèle en vérifiant l'égalité des prédictions, provoquer et diagnostiquer un
**train/serving skew**, détecter une **dérive** par un test statistique, et produire une **fiche modèle**.

Aucune donnée téléchargée : jeu synthétique. Les artefacts sont écrits dans un dossier temporaire puis
supprimés (aucun fichier binaire n'est laissé dans le dépôt).

In [1]:
import json, os, tempfile
import numpy as np
import joblib
from scipy import stats
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

rng = np.random.default_rng(0)
print("joblib", joblib.__version__)

joblib 1.6.0


## 1. Une Pipeline sans fuite

Le prétraitement (ici une standardisation) est appris **uniquement sur le train** et rejoué à l'identique
sur le test : la `Pipeline` matérialise cette frontière (module 1 ; module 7, §4.1).

In [2]:
X, y = make_classification(
    n_samples=1500, n_features=6, n_informative=5, n_redundant=0,
    class_sep=2.5, random_state=0,
)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

modele = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=0))
modele.fit(X_tr, y_tr)

acc = accuracy_score(y_te, modele.predict(X_te))
print(f"Accuracy test : {acc:.3f}")
assert acc > 0.8, "jeu séparable : l'accuracy doit être correcte"

# Le scaler a appris ses statistiques sur le TRAIN uniquement.
scaler = modele.named_steps["standardscaler"]
assert np.allclose(scaler.mean_, X_tr.mean(axis=0)), "les moyennes viennent du train, pas du test"
print("-> statistiques de standardisation issues du train (aucune fuite du test).")

Accuracy test : 0.989
-> statistiques de standardisation issues du train (aucune fuite du test).


## 2. Persistance : sauvegarder, recharger, vérifier

Un modèle rechargé doit produire **exactement** les mêmes prédictions. On sérialise la Pipeline complète
(prétraitement inclus) : recharger les poids sans leur prétraitement est une erreur classique
(module 7, §2).

In [3]:
dossier = tempfile.mkdtemp()
chemin = os.path.join(dossier, "modele.joblib")

joblib.dump(modele, chemin)
modele_recharge = joblib.load(chemin)

pred_avant = modele.predict(X_te)
pred_apres = modele_recharge.predict(X_te)
proba_avant = modele.predict_proba(X_te)
proba_apres = modele_recharge.predict_proba(X_te)

print("taille du fichier :", os.path.getsize(chemin), "octets")
assert np.array_equal(pred_avant, pred_apres), "prédictions identiques après rechargement"
assert np.allclose(proba_avant, proba_apres), "probabilités identiques après rechargement"
print("-> le modèle rechargé est équivalent à l'original.")

os.remove(chemin); os.rmdir(dossier)   # nettoyage : aucun artefact laissé

taille du fichier : 1537 octets
-> le modèle rechargé est équivalent à l'original.


## 3. Train/serving skew : le piège n°1 en production

Si la donnée servie diffère de celle de l'entraînement — ici une **feature dans une mauvaise unité**
(multipliée par 10) — le modèle se dégrade **silencieusement** (module 7, §4.2). Le code ne plante pas ;
seule la performance chute.

In [4]:
X_te_correct = X_te.copy()
X_te_skew = X_te.copy()
X_te_skew[:, 0] *= 10.0        # ex. : une feature envoyée en centimètres au lieu de mètres

acc_correct = accuracy_score(y_te, modele.predict(X_te_correct))
acc_skew = accuracy_score(y_te, modele.predict(X_te_skew))
print(f"Accuracy (donnée correcte) : {acc_correct:.3f}")
print(f"Accuracy (donnée décalée)  : {acc_skew:.3f}")
assert acc_skew < acc_correct, "le skew doit dégrader la performance"
print("-> même code, mêmes poids : une simple erreur d'unité fait chuter la qualité.")

Accuracy (donnée correcte) : 0.989
Accuracy (donnée décalée)  : 0.640
-> même code, mêmes poids : une simple erreur d'unité fait chuter la qualité.


## 4. Détecter une dérive par un test statistique

On surveille la distribution d'une feature entre une **référence** (entraînement) et un flux de
**production**. Le test de Kolmogorov–Smirnov compare deux échantillons ; une p-value faible signale une
distribution différente. Rappel (module 7, §7.1) : une dérive détectée est un **déclencheur d'enquête**,
pas une preuve de baisse de performance.

In [5]:
reference = rng.normal(0.0, 1.0, size=2000)      # distribution d'entraînement
prod_stable = rng.normal(0.0, 1.0, size=2000)     # même distribution -> pas de dérive
prod_derive = rng.normal(0.8, 1.0, size=2000)     # moyenne décalée -> dérive

ks_stable = stats.ks_2samp(reference, prod_stable)
ks_derive = stats.ks_2samp(reference, prod_derive)
print(f"Feature stable  : statistique={ks_stable.statistic:.3f}, p-value={ks_stable.pvalue:.3g}")
print(f"Feature dérivée : statistique={ks_derive.statistic:.3f}, p-value={ks_derive.pvalue:.3g}")

assert ks_derive.pvalue < 0.05, "la dérive nette doit être détectée (p-value faible)"
assert ks_derive.pvalue < ks_stable.pvalue, "la dérive présente une évidence plus forte que le flux stable"
print("-> la feature décalée déclenche l'alerte ; la feature stable non.")

Feature stable  : statistique=0.018, p-value=0.902
Feature dérivée : statistique=0.339, p-value=3.18e-102
-> la feature décalée déclenche l'alerte ; la feature stable non.


## 5. Une fiche modèle (*model card*) minimale

La fiche modèle documente objectif, données, métrique, seuil et **limites** (module 2, §14 ; module 8, §8).
On l'assemble ici comme un objet versionnable, sérialisé en JSON.

In [6]:
fiche_modele = {
    "nom": "classifieur_demo",
    "version": "1.0.0",
    "date": "2026-09-05",
    "donnees": "jeu synthétique make_classification (1500 lignes, 6 features)",
    "metrique_test": {"accuracy": round(float(acc), 3)},
    "taille_train": int(len(y_tr)),
    "seuil_decision": 0.5,
    "usages_prevus": "démonstration pédagogique du module 7",
    "usages_interdits": "toute décision réelle sur des personnes",
    "limites": [
        "données synthétiques non représentatives d'un cas réel",
        "aucune évaluation par sous-groupe",
        "seuil non calibré sur un coût d'erreur métier",
    ],
}
print(json.dumps(fiche_modele, ensure_ascii=False, indent=2))

requis = {"nom", "version", "metrique_test", "seuil_decision", "limites"}
assert requis.issubset(fiche_modele), "la fiche doit contenir les champs essentiels"
assert fiche_modele["limites"], "les limites doivent être explicites, jamais vides" 

{
  "nom": "classifieur_demo",
  "version": "1.0.0",
  "date": "2026-09-05",
  "donnees": "jeu synthétique make_classification (1500 lignes, 6 features)",
  "metrique_test": {
    "accuracy": 0.989
  },
  "taille_train": 1050,
  "seuil_decision": 0.5,
  "usages_prevus": "démonstration pédagogique du module 7",
  "usages_interdits": "toute décision réelle sur des personnes",
  "limites": [
    "données synthétiques non représentatives d'un cas réel",
    "aucune évaluation par sous-groupe",
    "seuil non calibré sur un coût d'erreur métier"
  ]
}


## 6. Exercice guidé — vérifier une sérialisation

Écrivez `sauvegarde_fidele(modele, X)` : sauvegarde le modèle dans un fichier temporaire, le recharge, et
renvoie `True` si les prédictions sont identiques avant/après (puis nettoie le fichier).
**Critère de réussite** : renvoie `True` pour la Pipeline entraînée.

In [7]:
def sauvegarde_fidele(modele, X):
    # TODO :
    #   1. dumper le modèle avec joblib dans un fichier temporaire
    #   2. le recharger
    #   3. comparer modele.predict(X) et recharge.predict(X)
    #   4. supprimer le fichier et renvoyer le booléen
    raise NotImplementedError("À compléter")

# assert sauvegarde_fidele(modele, X_te) is True

### Solution

In [8]:
def sauvegarde_fidele(modele, X):
    fd, chemin = tempfile.mkstemp(suffix=".joblib")
    os.close(fd)
    try:
        joblib.dump(modele, chemin)
        recharge = joblib.load(chemin)
        return bool(np.array_equal(modele.predict(X), recharge.predict(X)))
    finally:
        os.remove(chemin)

assert sauvegarde_fidele(modele, X_te) is True
print("OK : la sérialisation préserve les prédictions.")

OK : la sérialisation préserve les prédictions.


## Récapitulatif

- La **Pipeline** empêche les fuites : le prétraitement est appris sur le train, rejoué sur le test.
- **Persister le modèle complet** (prétraitement inclus) et vérifier l'égalité des prédictions au rechargement.
- Le **train/serving skew** dégrade silencieusement : partager le code de transformation entre entraînement et service.
- Une **dérive** se surveille (ex. test KS) mais ne prouve pas seule une baisse de performance.
- Une **fiche modèle** rend explicites usages, seuil et limites.

Suite : [Module 8 — éthique, sécurité & régulation](../08_ethique_securite_regulation/cours_ethique_securite_regulation.md).